In [1]:
"""
耐受状态转移模型演示
"""

import sys
import os
from pathlib import Path

# 获取当前 notebook 所在的目录（Jupyter 中）
try:
    # 尝试获取 __file__（在 .py 文件中有效）
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # 在 Jupyter notebook 中，使用当前工作目录
    current_dir = os.getcwd()
    
# 将当前目录添加到系统路径
sys.path.append(current_dir)

from models.tolerant_model import TolerantStateTransitionModel
from models.base_model import EventType

In [2]:
def demo_tolerant_model():
    """演示耐受状态转移模型"""
    print("=" * 80)
    print("耐受状态转移模型演示")
    print("=" * 80)
    
    # 创建模型实例 - n=2, k=2, m=3
    model = TolerantStateTransitionModel(n=2, k=2, m=3)
    
    n_val = model.model_params['n']
    k_val = model.model_params['k']
    m_val = model.model_params['m']
    
    print(f"\n📊 模型信息:")
    print(f"   参数: n={n_val}, k={k_val}, m={m_val}")
    print(f"   初始状态: (1,0,0) 不健康")
    print(f"   可用干预:")
    print(f"      - 干预A: (0,0,1) 🔵")
    print(f"      - 干预B: (0,1,0) 🟢")
    print(f"      - 无干预: (0,0,0) ⚪")
    
    print(f"\n📖 状态说明:")
    print(f"   (1,0,0) → 不健康状态（包括耐受状态）")
    print(f"   (0,0,0) → 健康状态")
    
    print(f"\n📖 事件说明:")
    print(f"   E4a (完全转移): {n_val+k_val}次干预A + {m_val}次干预B → 稳定健康")
    print(f"   E4b (耐受回退): {n_val+k_val}次干预A → 产生耐受，回到不健康")
    print(f"   E4c (部分转移): {n_val}次干预A → 健康（但不稳定）")
    print(f"   E4d (无有效转移): 其他情况 → 不健康")
    
    # 场景1: 部分转移 (连续n次干预A)
    print("\n" + "=" * 80)
    print(f"场景1: 部分转移 - 连续 {n_val} 次干预A")
    print("=" * 80)
    
    interventions_scene1 = [(0, 0, 1)] * n_val
    
    result1 = model.simulate(interventions_scene1, return_details=True)
    _print_results(result1, model, title=f"连续{n_val}次干预A → 健康（但不稳定）")
    
    # 场景2: 耐受回退 (连续n+k次干预A)
    print("\n" + "=" * 80)
    print(f"场景2: 耐受回退 - 连续 {n_val + k_val} 次干预A")
    print("=" * 80)
    
    interventions_scene2 = [(0, 0, 1)] * (n_val + k_val)
    
    result2 = model.simulate(interventions_scene2, return_details=True)
    result2 = model.simulate(interventions_scene2, return_details=True)
    _print_results(result2, model, title=f"连续{n_val + k_val}次干预A → 产生耐受，回到不健康")
    
    # 场景3: 完全转移 (n+k次干预A + m次干预B)
    print("\n" + "=" * 80)
    print(f"场景3: 完全转移 - {n_val + k_val}次干预A + {m_val}次干预B")
    print("=" * 80)
    
    interventions_scene3 = [(0, 0, 1)] * (n_val + k_val) + [(0, 1, 0)] * m_val
    
    result3 = model.simulate(interventions_scene3, return_details=True)
    _print_results(result3, model, title=f"{n_val + k_val}次干预A + {m_val}次干预B → 稳定健康")
    
    # 场景4: 耐受后使用干预B进行完全转移
    print("\n" + "=" * 80)
    print(f"场景4: 耐受后使用干预B进行完全转移")
    print("=" * 80)
    
    interventions_scene4 = (
        [(0, 0, 1)] * (n_val + k_val) +  # 产生耐受
        [(0, 1, 0)] * m_val               # 干预B达到完全转移
    )
    
    result4 = model.simulate(interventions_scene4, return_details=True)
    _print_results(result4, model, title="产生耐受后，连续干预B → 稳定健康")
    
    # 场景5: 部分转移后停止干预（不稳定）
    print("\n" + "=" * 80)
    print(f"场景5: 部分转移后停止干预 - 不稳定健康")
    print("=" * 80)
    
    interventions_scene5 = (
        [(0, 0, 1)] * n_val +   # 部分转移，达到健康
        [(0, 0, 0)] * 3         # 停止干预
    )
    
    result5 = model.simulate(interventions_scene5, return_details=True)
    _print_results(result5, model, title="部分转移后停止干预 → 注意：健康状态不稳定")


def demo_parameter_comparison():
    """参数对比演示"""
    print("\n" + "=" * 80)
    print("参数对比：不同参数设置的影响")
    print("=" * 80)
    
    param_sets = [
        {"n": 2, "k": 2, "m": 3, "name": "n=2, k=2, m=3"},
        {"n": 3, "k": 2, "m": 4, "name": "n=3, k=2, m=4"},
        {"n": 2, "k": 3, "m": 3, "name": "n=2, k=3, m=3"},
    ]
    
    for params in param_sets:
        model = TolerantStateTransitionModel(
            n=params["n"], 
            k=params["k"], 
            m=params["m"]
        )
        
        print(f"\n{params['name']}:")
        print(f"   部分转移需要: {params['n']}次干预A")
        print(f"   耐受需要: {params['n'] + params['k']}次干预A")
        print(f"   耐受后完全转移需要: {params['m']}次干预B")


def demo_tolerant_visualization():
    """可视化耐受状态转移过程"""
    print("\n" + "=" * 80)
    print("耐受状态转移过程可视化")
    print("=" * 80)
    
    model = TolerantStateTransitionModel(n=2, k=2, m=3)
    n_val = model.model_params['n']
    k_val = model.model_params['k']
    m_val = model.model_params['m']
    
    print(f"\n参数: n={n_val}, k={k_val}, m={m_val}")
    print(f"\n状态转移路径:")
    print("-" * 60)
    print(f"1. 初始状态: (1,0,0) 不健康")
    print(f"   ↓")
    print(f"2. 连续 {n_val} 次干预A: (0,0,0) 健康（部分转移，不稳定）")
    print(f"   ↓")
    print(f"3. 继续干预A {k_val} 步: 产生耐受")
    print(f"   ↓")
    print(f"4. 耐受状态: (1,0,0) 不健康")
    print(f"   ↓")
    print(f"5. 连续 {m_val} 次干预B: (0,0,0) 稳定健康（完全转移）")
    print("-" * 60)
    
    print(f"\n💡 关键特性:")
    print(f"   - 部分转移后的健康状态是不稳定的")
    print(f"   - 产生耐受后需要不同的干预B来恢复")
    print(f"   - 联合使用干预A和干预B才能达到稳定健康")


def _print_results(result: dict, model, title: str = ""):
    """打印模拟结果"""
    if title:
        print(f"\n{title}")
    
    print(f"\n干预序列:")
    for t, inter in enumerate(result["interventions"], 1):
        if inter == model.INTERVENTION_A:
            print(f"   t={t:2d}: {inter} 🔵 干预A")
        elif inter == model.INTERVENTION_B:
            print(f"   t={t:2d}: {inter} 🟢 干预B")
        else:
            print(f"   t={t:2d}: {inter} ⚪ 无干预")
    
    print(f"\n状态演变:")
    print("-" * 80)
    print(f"{'步数':<6} {'可观测状态':<20} {'状态说明':<15} {'事件':<40}")
    print("-" * 80)
    
    for i in range(len(result["states"])):
        state = result["states"][i]
        
        if state == model.HEALTHY:
            state_name = "健康"
        else:
            state_name = "不健康"
        
        if i == 0:
            event_str = "-"
        else:
            event = result["events"][i-1]
            if event == EventType.COMPLETE_TRANSITION:
                event_str = "E4a (完全转移) ⭐"
            elif event == EventType.TOLERANCE_REVERSION:
                event_str = "E4b (耐受回退)"
            elif event == EventType.PARTIAL_TRANSITION:
                event_str = "E4c (部分转移)"
            else:
                event_str = "E4d (无有效转移)"
        
        display_state = f"{state} ({state_name})"
        print(f"{i:<6} {display_state:<20} {state_name:<15} {event_str:<40}")
    
    # 显示内部状态
    internal = model.get_internal_state()
    if internal["is_tolerant"]:
        print(f"\n⚠️  已产生耐受，需要干预B才能恢复")
    if internal["is_in_partial_state"]:
        print(f"\n⚠️  处于部分转移状态（健康但不稳定）")


In [3]:
if __name__ == "__main__":
    demo_tolerant_model()
    demo_parameter_comparison()
    demo_tolerant_visualization()

耐受状态转移模型演示

📊 模型信息:
   参数: n=2, k=2, m=3
   初始状态: (1,0,0) 不健康
   可用干预:
      - 干预A: (0,0,1) 🔵
      - 干预B: (0,1,0) 🟢
      - 无干预: (0,0,0) ⚪

📖 状态说明:
   (1,0,0) → 不健康状态（包括耐受状态）
   (0,0,0) → 健康状态

📖 事件说明:
   E4a (完全转移): 4次干预A + 3次干预B → 稳定健康
   E4b (耐受回退): 4次干预A → 产生耐受，回到不健康
   E4c (部分转移): 2次干预A → 健康（但不稳定）
   E4d (无有效转移): 其他情况 → 不健康

场景1: 部分转移 - 连续 2 次干预A

连续2次干预A → 健康（但不稳定）

干预序列:
   t= 1: (0, 0, 1) 🔵 干预A
   t= 2: (0, 0, 1) 🔵 干预A

状态演变:
--------------------------------------------------------------------------------
步数     可观测状态                状态说明            事件                                      
--------------------------------------------------------------------------------
0      (1, 0, 0) (不健康)      不健康             -                                       
1      (1, 0, 0) (不健康)      不健康             E4d (无有效转移)                             
2      (0, 0, 0) (健康)       健康              E4c (部分转移)                              

⚠️  处于部分转移状态（健康但不稳定）

场景2: 耐受回退 - 连续 4 次干预A

连续4次干预A → 产生耐